# 智能故事生成器 · StoryCreat Agent

> 写作 Agent × 独立审稿 Agent 的闭环故事生成 Demo

本 Notebook 演示一个可运行、可评测的智能故事生成器：用户输入创作需求，**写作 Agent** 生成初稿，**独立审稿 Agent** 依据规则与评分标准判断是否与需求相符；未通过时，写作 Agent 按审稿意见自动修订，直到通过或达到轮次上限。

## 作者信息

- **姓名**：Noob9331x
- **GitHub**：[@Noob9331x](https://github.com/Noob9331x)
- **日期**：2026-11-02
- **运行环境**：Python 3.14.5 · hello-agents 1.0.0 · JupyterLab

> 说明：本 Notebook 与项目中的 CLI、FastAPI 服务、原生前端共用同一套 `src/` 业务代码，不是独立的玩具示例。


## 项目简介

### 解决什么问题？

单次 LLM 调用“一次成稿”质量不稳定，缺少质量门禁；而让同一个 Agent 自我反思，又容易“自写自夸”。本项目把创作与审稿拆成两个独立 Agent，并加入确定性的规则层，形成可量化、可追溯的闭环。

### 核心能力

- **三类体裁**：小说 `novel`、剧本 `script`、诗歌 `poem`，各有独立的结构约束与提示词。
- **双 Agent 协作**：写作 Agent 只负责创作/修订，审稿 Agent 只负责评分与给修改指令，二者不共享对话上下文。
- **规则层保底**：字数、体裁结构、必含/禁含元素、元话语、复读、Markdown 残留等确定性检查先于 LLM 执行。
- **自动修订闭环**：未通过时携带原始需求、当前稿、问题清单和修订指令重写，最多 3 轮。
- **最佳稿保留**：达到轮次上限时返回历史最高分稿，而不是无条件返回最后一稿。
- **可降级**：审稿 JSON 无法解析或 LLM 异常时，降级为仅规则层判定，并标记 `degraded`。
- **可追溯**：每轮草稿、审稿 JSON、耗时、token 用量落盘到 `outputs/runs/<run_id>/`。
- **多入口**：Notebook、CLI（`main.py`）、HTTP API / 网页前端共用同一套编排逻辑。

### 架构

```text
用户需求 StoryRequest
        │
        ▼
┌───────────────┐   初稿 / 修订稿    ┌─────────────────────────┐
│  WriterAgent  │ ─────────────────▶ │ 规则层 run_rule_review   │
│  （写作）      │                    │ 字数/格式/必含/禁含/重复  │
└───────────────┘                    └────────────┬────────────┘
        ▲                                         │ 规则问题
        │ 修改意见 + 问题清单                        ▼
        │                              ┌─────────────────────────┐
        └───────────────────────────── │ ReviewerAgent（LLM 审稿）│
                                       │ 四维评分 + 问题 + 指令    │
                                       └────────────┬────────────┘
                                                    ▼
                                   合并问题 → 判定是否通过 → 落盘
```

通过条件（同时满足）：

1. `review.approved == true`
2. 加权总分 ≥ `APPROVE_SCORE`（默认 80）
3. `relevance` ≥ `MIN_RELEVANCE_SCORE`（默认 80）
4. 合并后不存在 `fatal` 问题

加权公式：`relevance × 0.4 + completeness × 0.2 + genre_format × 0.2 + style × 0.2`。


## 第 2 部分：环境配置

### 安装依赖

本项目在 Python 3.14.5 上验证。推荐直接安装项目依赖，而不是使用模板中的 `hello-agents[all]`：**hello-agents 1.0.0 并未提供 `[all]` extra**，直接安装 `requirements.txt` 更可控。

```bash
pip install -r requirements.txt
```

### 配置模型

复制 `.env.example` 为 `.env`，填写任意 OpenAI 兼容接口：

```ini
LLM_MODEL_ID=glm-4-flash
LLM_API_KEY=your-api-key-here
LLM_BASE_URL=https://open.bigmodel.cn/api/paas/v4/
```

> 当前仓库的 `.env` 为空时，本 Notebook 会跳过真实 LLM 调用，但仍会运行规则层、离线工具演示和完整流程回退演示。


In [1]:
# 如需在全新环境安装依赖，取消下一行注释后运行
# %pip install -q -r requirements.txt

from __future__ import annotations

import json
import platform
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path
from types import SimpleNamespace

# 保证从项目根目录导入 src（Notebook 位于项目根目录）
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env")

import hello_agents
from src.config import get_settings
from src.models import RunStatus, StoryRequest, TokenUsage

settings = get_settings()

print("Python：", platform.python_version())
print("hello-agents：", hello_agents.__version__)
print("项目根目录：", PROJECT_ROOT)
print("LLM 已配置：", settings.llm_configured)
print("模型：", settings.llm_model_id or "未配置")
print("通过分数线：", settings.approve_score, "| 默认轮次：", settings.max_rounds_default)


Python： 3.14.5
hello-agents： 1.0.0
项目根目录： E:\hello-agents\Co-creation-projects\Noob9331x-StoryCreat-Agent
LLM 已配置： True
模型： deepseek-flash
通过分数线： 80 | 默认轮次： 2


## 第 3 部分：工具定义

### 为什么需要工具？

审稿中的一部分判断是**确定性的**：字数是否达标、剧本有没有场次、诗歌有没有 8 行、必须元素是否出现、禁止元素是否混入。把这些检查封装成工具，可以让 Agent 在需要时调用，也可以在不调用 LLM 的情况下单独测试。

### hello-agents 1.0 的工具协议

模板中的 `from hello_agents.tools import BaseTool` 在 1.0.0 中并不存在。当前版本使用：

- `Tool`：工具基类；
- `ToolParameter`：声明参数；
- `ToolResponse.success / partial / error`：统一返回结构；
- `run(parameters: dict) -> ToolResponse`：同步执行入口。

下面把项目里的 `src.rules.run_rule_review` 封装为 `StoryRuleTool`。


In [2]:
from hello_agents.tools import Tool, ToolParameter, ToolResponse

from src.rules import count_chars, run_rule_review


class StoryRuleTool(Tool):
    """故事规则检查工具：包装确定性的规则层审稿。"""

    def __init__(self, request: StoryRequest) -> None:
        """绑定一次创作请求，后续检查都围绕该请求的约束进行。

        Args:
            request: 创作请求，提供目标长度、体裁、必含/禁含元素等约束。
        """
        super().__init__(
            name="story_rule_check",
            description=(
                "离线检查故事稿件：字数、体裁结构、必含/禁含元素、"
                "元话语、复读与 Markdown 格式。"
            ),
        )
        self.request = request

    def get_parameters(self) -> list[ToolParameter]:
        """声明工具参数。"""
        return [
            ToolParameter(
                name="text",
                type="string",
                description="待检查的故事稿件全文",
                required=True,
            )
        ]

    def run(self, parameters: dict) -> ToolResponse:
        """执行规则检查，返回结构化结果。"""
        text = str(parameters.get("text", "")).strip()
        if not text:
            return ToolResponse.error(
                code="INVALID_INPUT",
                message="text 不能为空。",
            )

        issues = run_rule_review(self.request, text)
        payload = [issue.model_dump(mode="json") for issue in issues]
        if not issues:
            return ToolResponse.success(
                text="规则检查通过：未发现硬性问题。",
                data={
                    "issue_count": 0,
                    "char_count": count_chars(text),
                    "issues": [],
                },
            )

        lines = [f"[{issue.severity.value}] {issue.problem}" for issue in issues]
        return ToolResponse.partial(
            text=f"发现 {len(issues)} 条规则问题：\n" + "\n".join(lines),
            data={
                "issue_count": len(issues),
                "char_count": count_chars(text),
                "issues": payload,
            },
        )


### 直接运行工具

先构造一条小说请求和一段明显过短的稿件，观察规则层的硬性检查结果。这个演示**不需要 API Key**。


In [3]:
sample_request = StoryRequest(
    prompt="写一个雨夜便利店的故事：夜班店员遇到一位寻找失散女儿的陌生老人，结尾两人达成和解。",
    genre="novel",
    style="温暖克制，细节真实",
    target_length=800,
    must_include=["雨", "便利店", "老人"],
    max_rounds=1,
)

short_draft = "雨夜，便利店的灯还亮着。"

rule_tool = StoryRuleTool(sample_request)
tool_response = rule_tool.run_with_timing({"text": short_draft})

print(tool_response.to_dict()["status"])
print(tool_response.text)
print("\n结构化数据：")
print(json.dumps(tool_response.data, ensure_ascii=False, indent=2))


partial
发现 2 条规则问题：
[fatal] 正文仅 12 字，低于目标长度 800 字的 60%（480 字）。
[fatal] 缺少用户要求的必须元素「老人」。

结构化数据：
{
  "issue_count": 2,
  "char_count": 12,
  "issues": [
    {
      "severity": "fatal",
      "category": "length",
      "quote": "雨夜，便利店的灯还亮着。",
      "problem": "正文仅 12 字，低于目标长度 800 字的 60%（480 字）。",
      "suggestion": "补写场景、人物行动或心理描写，使篇幅达到目标长度的 85% 以上。"
    },
    {
      "severity": "fatal",
      "category": "keyword",
      "quote": "老人",
      "problem": "缺少用户要求的必须元素「老人」。",
      "suggestion": "在正文中自然融入「老人」，并确保其不是一笔带过。"
    }
  ]
}


## 第 4 部分：智能体构建

本项目的生产链路有两层：

1. **`SimpleAgent` + 工具**：演示 hello-agents 的标准 Agent 用法，把规则检查工具挂到 Agent 上；
2. **`WriterAgent` / `ReviewerAgent` + `StoryPipeline`**：项目实际的写作—审稿—修订编排，写作与审稿使用独立的 LLM 实例和提示词。

> 注意：`SimpleAgent` 的工具调用依赖模型的 function calling 能力；如果所选模型不支持工具调用，Agent 仍可直接使用，工具本身也可以像上一节那样独立执行。


In [4]:
from hello_agents import HelloAgentsLLM, SimpleAgent

from src.agents.base import build_agent_config

assistant = None
if settings.llm_configured:
    assistant_llm = HelloAgentsLLM(
        model=settings.llm_model_id,
        api_key=settings.llm_api_key,
        base_url=settings.llm_base_url,
        temperature=settings.writer_temperature,
        timeout=settings.llm_timeout,
    )
    assistant = SimpleAgent(
        name="故事质检助手",
        llm=assistant_llm,
        system_prompt=(
            "你是一名严谨的故事质检助手。"
            "需要检查稿件时，请调用 story_rule_check 工具，"
            "再根据工具返回的问题给出简短、可执行的修改建议。"
        ),
        config=build_agent_config(),
        enable_tool_calling=True,
    )
    assistant.add_tool(StoryRuleTool(sample_request))
    print("已注册工具：", assistant.tool_registry.list_tools())
    print(assistant.tool_registry.get_tools_description())
else:
    print("未配置 LLM，跳过 SimpleAgent 实例化。")
    print("规则检查工具仍可直接调用，见上一节。")


✅ 工具 'story_rule_check' 已注册。
已注册工具： ['story_rule_check']
- story_rule_check: 离线检查故事稿件：字数、体裁结构、必含/禁含元素、元话语、复读与 Markdown 格式。


In [5]:
from src.agents.reviewer import ReviewerAgent
from src.agents.writer import WriterAgent
from src.pipeline import StoryPipeline
from src.storage import RunStorage

writer = None
reviewer = None
pipeline = None

if settings.llm_configured:
    # 写作与审稿使用独立 LLM 实例，避免共享对话上下文。
    writer_llm = HelloAgentsLLM(
        model=settings.llm_model_id,
        api_key=settings.llm_api_key,
        base_url=settings.llm_base_url,
        temperature=settings.writer_temperature,
        timeout=settings.llm_timeout,
    )
    reviewer_llm = HelloAgentsLLM(
        model=settings.llm_model_id,
        api_key=settings.llm_api_key,
        base_url=settings.llm_base_url,
        temperature=settings.reviewer_temperature,
        timeout=settings.llm_timeout,
    )
    writer = WriterAgent(writer_llm, settings)
    reviewer = ReviewerAgent(reviewer_llm, settings)
    pipeline = StoryPipeline(
        writer=writer,
        reviewer=reviewer,
        settings=settings,
        storage=RunStorage(settings.output_dir),
    )
    print("WriterAgent / ReviewerAgent / StoryPipeline 已创建。")
else:
    print("未配置 LLM：跳过真实 Agent 创建。")
    print("下一节仍会展示规则层和无 Key 的完整流程回退。")


WriterAgent / ReviewerAgent / StoryPipeline 已创建。


### 可选：让 `SimpleAgent` 调用工具

如果已配置支持 function calling 的模型，可以运行下面的可选演示。它会真实消耗一次模型调用；失败通常意味着模型不支持工具调用，不影响后面的流水线演示。


In [6]:
if assistant is not None:
    try:
        answer = assistant.run(
            "请调用 story_rule_check 工具检查下面这段稿件，并用一句话总结问题：\n"
            + short_draft
        )
        print(answer)
    except Exception as exc:  # noqa: BLE001 - 演示环境允许模型/工具能力差异
        print("SimpleAgent 工具调用未成功：", type(exc).__name__, exc)
else:
    print("未配置 LLM，跳过 agent.run 演示。")


检查结果：稿件只有一句话（12 字），远少于 800 字的下限，并且缺少必须出现的人物“老人”。

一句话总结：这段稿件太短，且没有出现必须包含的元素“老人”，需扩写到不少于 480 字，并让“老人”出场。


## 第 5 部分：功能演示

本节分三个演示：

1. **离线规则审稿**：不调用模型，展示规则层能发现什么问题；
2. **真实双 Agent 闭环**：需要在 `.env` 中配置模型，展示生成、评分、修订与终稿；
3. **无 Key 离线回退**：用模拟 LLM 跑通 `StoryPipeline` 状态机，仅验证流程，不代表真实质量。

先定义统一的 Markdown 渲染函数，避免每个演示重复写展示代码。


In [7]:
from IPython.display import Markdown, display

from src.pipeline import rule_only_scores

STATUS_LABELS = {
    "approved": "已通过",
    "max_rounds_reached": "达到最大轮次",
    "degraded": "审稿降级",
    "failed": "生成失败",
}


def _cell_text(value: object, limit: int = 80) -> str:
    """把任意值压缩成适合放进 Markdown 表格的一行文本。"""
    text = str(value if value is not None else "").replace("|", "\\|").replace("\n", " ")
    return text[:limit] + ("…" if len(text) > limit else "")


def render_issues(issues: list, limit: int = 10) -> None:
    """把审稿问题渲染成 Markdown 表格。"""
    if not issues:
        display(Markdown("**本轮无审稿问题。**"))
        return
    rows = [
        "| 级别 | 类别 | 原文引用 | 问题 | 建议 |",
        "|---|---|---|---|---|",
    ]
    for issue in issues[:limit]:
        rows.append(
            "| "
            + " | ".join(
                [
                    issue.severity.value,
                    _cell_text(issue.category, 30),
                    _cell_text(issue.quote, 40),
                    _cell_text(issue.problem, 90),
                    _cell_text(issue.suggestion, 90),
                ]
            )
            + " |"
        )
    display(Markdown("\n".join(rows)))


def render_result(result) -> None:
    """渲染 GenerationResult：状态、逐轮评分、审稿问题与终稿。"""
    label = STATUS_LABELS.get(result.status.value, result.status.value)
    best_round = result.best_round if result.best_round is not None else "-"
    display(Markdown(f"### 运行结果：`{result.status.value}`（{label}）"))
    display(
        Markdown(
            f"- **运行 ID**：`{result.run_id}`\n"
            f"- **采用轮次**：{best_round} / {len(result.rounds)}\n"
            f"- **模型**：`{result.model or '未记录'}`\n"
            f"- **Token**：{result.usage.total_tokens} "
            f"(prompt {result.usage.prompt_tokens} / completion {result.usage.completion_tokens})\n"
            f"- **创建时间**：{result.created_at:%Y-%m-%d %H:%M:%S}"
        )
    )
    if result.error:
        display(Markdown(f"> 提示：{result.error}"))

    if not result.rounds:
        display(Markdown("**本次运行没有产生轮次记录。**"))
        return

    rows = [
        "| 轮次 | 总分 | 相关性 | 完整性 | 体裁 | 文风 | 通过 | 耗时(ms) | Token |",
        "|---:|---:|---:|---:|---:|---:|:---:|---:|---:|",
    ]
    for record in result.rounds:
        scores = (
            record.review.scores
            if record.review is not None
            else rule_only_scores(record.rule_issues)
        )
        rows.append(
            f"| {record.round} | {record.overall_score} | {scores.relevance} | "
            f"{scores.completeness} | {scores.genre_format} | {scores.style} | "
            f"{'✅' if record.approved else ('⚠️' if record.degraded else '❌')} | "
            f"{record.latency_ms} | {record.usage.total_tokens} |"
        )
    display(Markdown("#### 逐轮评分"))
    display(Markdown("\n".join(rows)))

    for record in result.rounds:
        flag = "已通过" if record.approved else ("降级" if record.degraded else "未通过")
        display(Markdown(f"#### 第 {record.round} 轮 · {record.overall_score} 分 · {flag}"))
        if record.review and record.review.summary:
            display(Markdown(f"> {record.review.summary}"))
        render_issues(record.merged_issues)

    display(Markdown("### 终稿"))
    display(Markdown(result.final_text or "（本次运行没有产出正文）"))


### 示例 1：离线规则审稿（无需 API Key）

对一段只有一句话的“稿件”执行规则检查。它应当触发**长度不足**的 `fatal` 问题；因为规则层不调用模型，所以结果完全确定、可复现。


In [8]:
print("=== 示例 1：离线规则审稿 ===")
print(f"稿件字符数：{count_chars(short_draft)} / 目标 {sample_request.target_length}")

demo_issues = run_rule_review(sample_request, short_draft)
print(f"命中规则问题：{len(demo_issues)} 条")
render_issues(demo_issues)

print("\n说明：规则层负责可机械判定的硬约束；语义是否跑题、文风是否合适，交给下一节的 LLM 审稿。")


=== 示例 1：离线规则审稿 ===
稿件字符数：12 / 目标 800
命中规则问题：2 条


| 级别 | 类别 | 原文引用 | 问题 | 建议 |
|---|---|---|---|---|
| fatal | length | 雨夜，便利店的灯还亮着。 | 正文仅 12 字，低于目标长度 800 字的 60%（480 字）。 | 补写场景、人物行动或心理描写，使篇幅达到目标长度的 85% 以上。 |
| fatal | keyword | 老人 | 缺少用户要求的必须元素「老人」。 | 在正文中自然融入「老人」，并确保其不是一笔带过。 |


说明：规则层负责可机械判定的硬约束；语义是否跑题、文风是否合适，交给下一节的 LLM 审稿。


### 示例 2：真实双 Agent 闭环（需要 API Key）

如果 `.env` 已配置，下面的代码会真实调用模型：

- 写作 Agent 生成初稿；
- 规则层 + 审稿 Agent 独立审稿；
- 未通过则按意见修订；
- 最终返回历史最高分稿，并展示逐轮评分与问题。

> 这一步会消耗 token，耗时取决于模型速度。默认 `max_rounds=2`，因此最多产生 2 轮写作 + 2 轮审稿。


In [9]:
print("=== 示例 2：真实双 Agent 闭环 ===")

if pipeline is None:
    print("未配置 LLM，跳过真实生成。")
    print("请复制 .env.example 为 .env，填写 LLM_MODEL_ID / LLM_API_KEY / LLM_BASE_URL，")
    print("重启 Jupyter kernel 后重新运行本 Notebook。")
else:
    real_request = sample_request.model_copy(update={"max_rounds": 2})
    started = time.perf_counter()
    real_result = pipeline.run(real_request)
    elapsed = time.perf_counter() - started
    print(f"端到端耗时：{elapsed:.1f} 秒")
    render_result(real_result)


=== 示例 2：真实双 Agent 闭环 ===


端到端耗时：15.9 秒


### 运行结果：`approved`（已通过）

- **运行 ID**：`20261102-202524-3823`
- **采用轮次**：1 / 1
- **模型**：`deepseek-flash`
- **Token**：3781 (prompt 1783 / completion 1998)
- **创建时间**：2026-11-02 20:25:24

#### 逐轮评分

| 轮次 | 总分 | 相关性 | 完整性 | 体裁 | 文风 | 通过 | 耗时(ms) | Token |
|---:|---:|---:|---:|---:|---:|:---:|---:|---:|
| 1 | 90 | 90 | 88 | 92 | 91 | ✅ | 15246 | 3781 |

#### 第 1 轮 · 90 分 · 已通过

> 稿件紧扣“雨夜便利店—夜班店员—寻找失散女儿的陌生老人”的核心设定，雨、便利店、老人三要素齐全，无禁止元素。小说结构完整，有起承转合，细节真实（关东煮汤、冷柜压缩机、压扁的纸杯、照片压出的水痕），文风温暖克制，仅结尾一处比喻略直白。篇幅约950—1000字，略超“约800字”；“两人达成和解”的实现偏单向，可略作加强。总体偏差轻微，可用。

| 级别 | 类别 | 原文引用 | 问题 | 建议 |
|---|---|---|---|---|
| minor | length | 雨从傍晚开始下，到夜里十一点还没停。 | 全文约950—1000字，超出“约800字”目标约两成，情节虽未有赘余，但仍略长于设定。 | 可压缩老人叙述身世的段落（如“我每年都出来找”“老伴去年走了”两处合并），把字数收束到800—850字区间。 |
| minor | relevance | “我知道她大概是怪我。”老人看着杯子里的水汽，“我那天要是牵着她就好了。” | “和解”的落点偏向老人单方面放下执念、店员给予善意接纳；两人之间此前并无分歧或摩擦，双向“达成和解”的意味略弱，更像单方的宽慰。 | 可在结尾补一个极轻的双向呼应，例如林晚也因这次相遇对某段旧事释然（前文商场走丢的回忆已埋伏笔），让“和解”落在两人身上而不只是老人一方。 |
| minor | style | 背影在路灯和雨丝之间慢慢变小，但走得不急，像一个终于知道有个地方可以去的人。 | “温暖克制”的整体基调把握得好，但此处比喻带有明显的解释与点题意味，稍微把情绪说破了，克制度略有折损。 | 改为只呈现动作与物象，如“背影在路灯和雨丝之间慢慢变小，走得不急”，把余味留给读者自行补足。 |

### 终稿

雨从傍晚开始下，到夜里十一点还没停。玻璃门被风推得轻响，便利店门口的塑料垫吸饱了水，踩上去发出闷闷的吱声。

林晚把最后一盒关东煮的汤添满，抬头看见一个老人站在门外。他没打伞，灰白的头发贴在额头上，两手空空，只是隔着玻璃往里看，目光在货架间来回移动，像在找什么东西。

她推开门：“进来避避雨吧。”

老人愣了一下，才跨进来，脚步很轻，鞋底带进来的水在地砖上留下一小片深色。他站在收银台前，没买东西，只是问：“姑娘，你见过一个女孩吗？这么高。”他抬手比到胸口的位置，“七岁，扎两个辫子，穿红毛衣。”

林晚摇头。她在这家店做了半年夜班，见过很多深夜进来的人，但没有一个小女孩。

老人点点头，像早就料到这个答案。他退到窗边的高脚凳坐下，把湿漉漉的手掌在裤子上擦了擦，然后从口袋里掏出一张照片，放在台面上。照片边角已经卷起，上面的女孩笑着，背后是一棵开花的树。

“三年前丢的。”他说，声音很平，“在城南的集市。我一转头，就不见了。”

林晚不知道该说什么。关东煮的锅咕嘟咕嘟响，冷柜的压缩机在角落里低声运转。她倒了一杯热水，放在老人手边。老人道了谢，捧着杯子，但没有喝。

“我每年都出来找。”他说，“走到哪儿算哪儿。有人跟我说，孩子长大了会认不出来。可我觉得我能认出来。她左耳后面有颗痣，米粒那么大。”

雨敲在玻璃上，密密的一片。老人继续说，他老伴去年走了，走之前说，别找了，找不到了。他答应了，可第二天还是收拾了包出门。

“我知道她大概是怪我。”老人看着杯子里的水汽，“我那天要是牵着她就好了。”

林晚想起自己小时候也走丢过一次，在商场里，只有几分钟。那几分钟里她的心一直跳，觉得世界忽然又大又空。后来她母亲找到她，第一件事是打了她一下，然后抱着她哭。

她看着老人，忽然说：“我不认识您女儿。可是您要是愿意，以后路过这儿，可以进来坐坐。我夜班，从十点到早上六点，都在。”

老人抬起头。

“不买也行。”林晚补了一句，觉得这话有点笨，“水是免费的。”

雨小了一些，变成细密的沙沙声。老人看着窗外，很久，然后他把照片收起来，小心地放回内袋，按了按。

“好。”他说。

他喝了那杯水，喝完把纸杯压扁，扔进旁边的垃圾桶。站起来的时候，他好像比刚才矮了一点，也松了一点。

“我姓周。”他说。

“林晚。”

老人朝她点点头，推门走进雨里。门关上的瞬间，风把外面的潮气送进来。林晚看见他没有撑伞，背影在路灯和雨丝之间慢慢变小，但走得不急，像一个终于知道有个地方可以去的人。

她把台面上被照片压出的一小圈水痕擦掉，重新添了一锅汤。

雨还在下。

### 示例 3：无 Key 离线回退（仅验证流程）

为了在**没有 API Key** 的环境中也能完整演示状态机，下面注入一个确定性的 `OfflineDemoLLM`：

- 写作请求返回固定稿件；
- 审稿请求返回固定 JSON（通过、90 分）；
- `StoryPipeline` 仍执行规则层、评分合并、落盘与最佳稿选择；
- 输出会明确标注为模拟数据。

> 这不是模型能力评估，只用于验证“生成 → 审稿 → 定稿”的工程链路。若已配置真实 LLM，本节会自动跳过。


In [10]:
print("=== 示例 3：无 Key 离线回退 ===")

if pipeline is not None:
    print("已配置真实 LLM，跳过离线模拟。")
else:
    offline_request = StoryRequest(
        prompt="写一个雨夜便利店的故事，主角是夜班店员，结尾要有和解。",
        genre="novel",
        style="温暖克制",
        target_length=200,
        must_include=["雨", "便利店"],
        max_rounds=2,
    )

    demo_draft = (
        "雨从傍晚开始下，便利店的玻璃门被风推得轻轻作响。林岚把最后一箱牛奶码上货架，"
        "抬头看见一个浑身湿透的老人站在门口，手里攥着一张被雨水泡皱的地址条。\n\n"
        "老人说他要找二十三年前在这里上夜班的女儿。林岚没有立刻回答，她给他倒了杯热水，"
        "把地址条摊在收银台上烘干，字迹一点点浮出来，竟是她母亲的名字。\n\n"
        "凌晨三点，雨停了。林岚关掉半排灯，陪老人坐在窗边等天亮。她第一次听他说起"
        "那些被家人刻意避开的往事，也第一次明白母亲为何总在雨夜多煮一碗面。"
    )

    def offline_review_json() -> str:
        """构造一份合法的审稿 JSON。"""
        return json.dumps(
            {
                "approved": True,
                "overall_score": 90,
                "scores": {
                    "relevance": 92,
                    "completeness": 90,
                    "genre_format": 90,
                    "style": 88,
                },
                "issues": [],
                "revision_instructions": [],
                "summary": (
                    "稿件紧扣雨夜便利店与和解主题，结构完整，人物关系清晰，"
                    "可以通过本轮审稿。"
                ),
            },
            ensure_ascii=False,
        )

    class OfflineDemoLLM:
        """仅用于离线演练的假 LLM。"""

        model = "offline-demo"
        provider = "offline"

        def __init__(self) -> None:
            self.calls = 0

        def invoke(self, messages, **kwargs):
            self.calls += 1
            system = messages[0]["content"] if messages else ""
            content = offline_review_json() if "审稿人" in system else demo_draft
            return SimpleNamespace(
                content=content,
                usage={
                    "prompt_tokens": 120,
                    "completion_tokens": 240,
                    "total_tokens": 360,
                },
                latency_ms=8,
            )

    offline_dir = Path(tempfile.mkdtemp(prefix="story-offline-"))
    offline_settings = settings.model_copy(
        update={
            "llm_model_id": "offline-demo",
            "llm_api_key": "offline-key",
            "llm_base_url": "http://localhost",
            "output_dir": offline_dir,
        }
    )
    offline_pipeline = StoryPipeline(
        writer=WriterAgent(OfflineDemoLLM(), offline_settings),
        reviewer=ReviewerAgent(OfflineDemoLLM(), offline_settings),
        settings=offline_settings,
        storage=RunStorage(offline_dir),
    )
    offline_result = offline_pipeline.run(offline_request)
    render_result(offline_result)
    print("\n⚠️ 以上为离线模拟输出，仅验证工程流程，不代表真实模型质量。")

    shutil.rmtree(offline_dir, ignore_errors=True)


=== 示例 3：无 Key 离线回退 ===
已配置真实 LLM，跳过离线模拟。


## 第 6 部分：性能评估

评估分两层：

### 1. 离线单元测试（不需要 API Key）

`tests/` 使用 `FakeLLM` 替换真实模型，覆盖：

- 数据模型与请求参数校验；
- 规则层审稿；
- 审稿 JSON 解析、重试与降级；
- 提示词构建；
- 流水线状态机：一次通过 / 修订通过 / 轮次耗尽 / 审稿降级 / 生成失败；
- FastAPI 接口与静态页。

### 2. 批量评测（需要 API Key）

```bash
python eval/run_eval.py --limit 3 --rounds 1   # 小规模验证
python eval/run_eval.py                        # 全部 9 条样例
python eval/run_eval.py --chart-only           # 用已有 CSV 重新绘图
```

输出：`outputs/eval_summary.csv` 与 `outputs/demo_result.png`。


In [11]:
print("=== 离线单元测试 ===")

completed = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--color=no"],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    timeout=300,
)
output = (completed.stdout or "").strip()
if completed.stderr:
    output += "\n" + completed.stderr.strip()
print(output[-3000:])
print("\n退出码：", completed.returncode)
print("说明：测试全部离线运行，不访问真实模型。")


=== 离线单元测试 ===


............................................                             [100%]
============================== warnings summary ===============================
.venv\Lib\site-packages\fastapi\testclient.py:1
  E:\hello-agents\Co-creation-projects\Noob9331x-StoryCreat-Agent\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
    from starlette.testclient import TestClient as TestClient  # noqa

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
44 passed, 1 warning in 3.35s

退出码： 0
说明：测试全部离线运行，不访问真实模型。


### 规则层微基准（离线）

下面用 9 条内置样例的请求参数，对同一段短稿重复执行规则层检查，观察纯规则路径的耗时。它不评估模型质量，只说明“无 LLM 时的确定性检查”开销很小。


In [12]:
from src.samples import load_samples

print("=== 规则层微基准（离线） ===")
samples = load_samples(settings.data_dir / "samples.json")
print(f"样例数量：{len(samples)}")

rows = [
    "| 用例 | 体裁 | 目标字数 | 规则问题数 | 耗时(ms) |",
    "|---|---|---:|---:|---:|",
]
total_seconds = 0.0
for sample in samples:
    request = StoryRequest(
        prompt=sample["prompt"],
        genre=sample["genre"],
        style=sample.get("style"),
        target_length=sample.get("target_length", 1200),
        must_include=sample.get("must_include", []),
        forbidden=sample.get("forbidden", []),
        max_rounds=sample.get("max_rounds", 2),
    )
    started = time.perf_counter()
    issues = run_rule_review(request, short_draft)
    elapsed_ms = (time.perf_counter() - started) * 1000
    total_seconds += elapsed_ms / 1000
    rows.append(
        f"| {sample.get('id', '-')} | {sample.get('genre', '-')} | "
        f"{request.target_length} | {len(issues)} | {elapsed_ms:.3f} |"
    )

display(Markdown("\n".join(rows)))
print(f"总耗时：{total_seconds * 1000:.2f} ms；平均：{total_seconds / max(len(samples), 1) * 1000:.3f} ms/条")


=== 规则层微基准（离线） ===
样例数量：9


| 用例 | 体裁 | 目标字数 | 规则问题数 | 耗时(ms) |
|---|---|---:|---:|---:|
| novel-rainy-store | novel | 800 | 2 | 0.031 |
| novel-bookshop-letter | novel | 1000 | 3 | 0.014 |
| novel-last-metro | novel | 900 | 3 | 0.010 |
| script-night-diner | script | 900 | 6 | 0.020 |
| script-hospital-corridor | script | 1000 | 5 | 0.014 |
| script-space-station | script | 1200 | 5 | 0.012 |
| poem-city-rain | poem | 320 | 3 | 0.010 |
| poem-hometown-river | poem | 400 | 4 | 0.011 |
| poem-winter-morning | poem | 360 | 4 | 0.010 |

总耗时：0.13 ms；平均：0.015 ms/条


### 查看已有评测产物

如果之前运行过 `eval/run_eval.py`，下面会直接展示汇总表与图表；否则会给出运行提示。真实评测结果取决于模型、样例和网络环境，本 Notebook 不伪造分数。


In [13]:
print("=== 批量评测产物 ===")
print("运行命令：python eval/run_eval.py --limit 9")
print("要求：已在 .env 中配置可用 LLM")
print(f"汇总路径：{settings.output_dir / 'eval_summary.csv'}")
print(f"图表路径：{settings.output_dir / 'demo_result.png'}")

summary_path = settings.output_dir / "eval_summary.csv"
chart_path = settings.output_dir / "demo_result.png"

if summary_path.exists():
    import csv

    with summary_path.open("r", encoding="utf-8-sig", newline="") as handle:
        summary_rows = list(csv.DictReader(handle))
    if summary_rows:
        columns = ["case_id", "genre", "status", "rounds", "final_score", "total_tokens"]
        table = [
            "| " + " | ".join(columns) + " |",
            "|" + "|".join(["---"] * len(columns)) + "|",
        ]
        for row in summary_rows[:10]:
            table.append(
                "| " + " | ".join(_cell_text(row.get(col, ""), 40) for col in columns) + " |"
            )
        display(Markdown("\n".join(table)))
    else:
        print("eval_summary.csv 存在，但没有数据行。")
else:
    print("尚未发现 eval_summary.csv，先配置模型并运行 eval 脚本。")

if chart_path.exists():
    from IPython.display import Image

    display(Image(filename=str(chart_path)))
else:
    print("尚未发现 demo_result.png。")


=== 批量评测产物 ===
运行命令：python eval/run_eval.py --limit 9
要求：已在 .env 中配置可用 LLM
汇总路径：E:\hello-agents\Co-creation-projects\Noob9331x-StoryCreat-Agent\outputs\eval_summary.csv
图表路径：E:\hello-agents\Co-creation-projects\Noob9331x-StoryCreat-Agent\outputs\demo_result.png
尚未发现 eval_summary.csv，先配置模型并运行 eval 脚本。
尚未发现 demo_result.png。


## 第 7 部分：总结与展望

### 实现的功能

- [x] 小说、剧本、诗歌三类体裁的提示词与结构约束；
- [x] `WriterAgent` 生成首稿、按审稿意见修订；
- [x] `ReviewerAgent` 独立审稿，输出四维评分、问题清单、修订指令的结构化 JSON；
- [x] 规则层确定性检查：长度、关键词、体裁结构、元话语、复读、Markdown 残留；
- [x] 综合判定：`approved` + 分数线 + 相关性 + 无 fatal；
- [x] 最多 3 轮自动修订，保留历史最高分稿；
- [x] 审稿解析失败 / LLM 异常时降级为规则层；
- [x] 每轮草稿、审稿、trace、result、final.md 落盘；
- [x] CLI、FastAPI、原生前端、批量评测脚本；
- [x] 44 个离线测试（FakeLLM，不访问网络）。

### 遇到的挑战与解决方案

| 挑战 | 解决方案 |
|---|---|
| LLM 不保证输出合法 JSON | 围栏清洗 → 大括号截取 → pydantic 校验 → 附带纠正提示重试 1 次 → 仍失败则降级 |
| 同一 Agent 自我反思容易“自写自夸” | 写作与审稿拆成两个独立 Agent，审稿人只接收原始需求和稿件，不共享写作历史 |
| 文笔好但跑题可能被放过 | 审稿提示词加入反讨好条款，评分中相关性权重 40%，规则层可硬否决 |
| 修订后分数反而下降 | 记录每轮总分，最终选择历史最高分稿；同分时优先更靠后、fatal 更少的轮次 |
| 无 API Key 时无法演示 | 规则层、工具、FakeLLM 测试均可离线；Notebook 提供明确标注的离线回退流程 |
| Windows / Python 3.14 兼容性 | 不依赖 Gradio/Streamlit；前端为原生静态页；依赖已在 3.14.5 验证 |
| 长文本生成超时或限流 | LLM 调用指数退避重试；接口返回 `failed`/`degraded` 而不是半截 JSON；前端展示 loading 与耗时 |

### 未来改进方向

- 增加 SSE 流式输出，实时展示写作过程；
- 支持长篇大纲 → 分章生成，加入人物/世界观一致性记忆；
- 增加更多体裁与模板（散文、儿童文学、广告文案）；
- 引入人工偏好数据，校准审稿 rubric 与模型评分偏差；
- 将运行记录迁移到 SQLite/PostgreSQL，支持多用户与并发任务；
- 增加版权相似度检测、敏感内容审核与更细粒度的可观测性。


## 附录：运行方式与项目结构

### 命令行

```bash
python main.py "写一个雨夜便利店的故事" --genre novel --length 800 --must "雨,便利店,老人" --rounds 2
python main.py "写一首城市雨夜的现代诗" --genre poem --length 320
python main.py --list
python main.py --serve --port 8000
```

### HTTP API

| 方法 | 路径 | 说明 |
|---|---|---|
| GET | `/api/health` | 健康检查与配置状态 |
| GET | `/api/samples` | 内置示例 |
| POST | `/api/generate` | 执行完整生成流程 |
| GET | `/api/runs?limit=20` | 历史运行摘要 |
| GET | `/api/runs/{run_id}` | 单次运行详情 |
| GET | `/api/runs/{run_id}/final.md` | 下载定稿（`?download=true`） |

### 目录结构

```text
.
├── main.py / main.ipynb      # CLI / Notebook 演示
├── src/
│   ├── api.py                # FastAPI
│   ├── pipeline.py           # 双 Agent 编排与判定
│   ├── agents/               # WriterAgent / ReviewerAgent
│   ├── rules.py              # 规则层
│   ├── prompts.py            # 提示词
│   ├── models.py             # pydantic 数据模型
│   └── storage.py            # 落盘与导出
├── frontend/index.html       # 原生单页前端
├── data/sample.csv           # 批量评测样例
├── eval/run_eval.py          # 批量评测与图表
└── tests/                    # 44 个离线测试
```

### 常见问题

- **提示“尚未配置 LLM”**：检查 `.env` 是否存在，以及 `LLM_MODEL_ID` / `LLM_API_KEY` / `LLM_BASE_URL` 是否填写；修改后重启 kernel。
- **审稿结果一直是 `degraded`**：查看 `outputs/runs/<run_id>/review_v*.json` 中的 `raw_text`，通常是模型没有按 JSON 输出或字段越界。
- **生成很慢**：减少 `target_length` 或 `max_rounds`；默认两轮最多 4 次模型调用。
- **规则层误判**：规则阈值与正则集中在 `src/rules.py`，可按体裁调整；调整后运行 `pytest tests/test_rules.py`。
- **前端页面空白**：确认 `frontend/index.html` 存在，并通过 `python main.py --serve` 访问 `/`，而不是直接双击 HTML。

> 本 Notebook 与项目代码保持同步；新增功能时请同步更新对应演示单元。
